# Structured Output Deep Dive

This notebook expands on the structured output labs in `01_openai/scripts`.
The goal is to understand the trade-offs between:

- typed parsing with Pydantic
- nested extraction
- strict JSON schema output
- recursive schemas
- moderation-style classification


In [1]:
import json
import sys
from enum import Enum
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "01_openai" / "scripts").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current notebook path")


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "01_openai" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI()
MODEL_NAME = "gpt-4o-2024-08-06"


## 1. Typed Parsing With Pydantic

This is the most ergonomic pattern when you want strongly typed Python objects back from the API.

In [2]:
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    confidence: float = Field(ge=0, le=1)
    short_reason: str


sentiment_response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {
            "role": "system",
            "content": "Analyze the user review and return a structured sentiment"
            "assessment.",
        },
        {
            "role": "user",
            "content": "The tutorial was clear and practical, but I wanted more"
            "advanced examples.",
        },
    ],
    text_format=SentimentAnalysis,
)

sentiment_response.output_parsed


SentimentAnalysis(sentiment='neutral', confidence=0.85, short_reason='The review expresses satisfaction with clarity and practicality, but mentions a desire for more advanced content.')

## 2. Nested Extraction

Nested models are useful when the response is still structured, but not flat.

In [3]:
class Person(BaseModel):
    name: str
    role: str


class CalendarEvent(BaseModel):
    title: str
    date: str
    location: str
    participants: list[Person]


event_response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "Extract the event details from the message."},
        {
            "role": "user",
            "content": "On Tuesday afternoon, Alice will present the new API demo at"
            "the Montreal office with Karim, who will handle the Q&A.",
        },
    ],
    text_format=CalendarEvent,
)

print(event_response.output_parsed.model_dump_json(indent=2))


{
  "title": "New API Demo Presentation",
  "date": "2023-04-04",
  "location": "Montreal office",
  "participants": [
    {
      "name": "Alice",
      "role": "Presenter"
    },
    {
      "name": "Karim",
      "role": "Q&A Handler"
    }
  ]
}


## 3. Strict JSON Schema

Use explicit JSON schema when you want raw JSON and tighter control over the payload shape.

In [4]:
project_schema = {
    "type": "object",
    "properties": {
        "projects": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "difficulty": {
                        "type": "string",
                        "enum": ["beginner", "intermediate", "advanced"],
                    },
                    "goal": {"type": "string"},
                },
                "required": ["title", "difficulty", "goal"],
                "additionalProperties": False,
            },
            "minItems": 2,
            "maxItems": 2,
        }
    },
    "required": ["projects"],
    "additionalProperties": False,
}

json_schema_response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {
            "role": "system",
            "content": "Return valid JSON only.",
        },
        {
            "role": "user",
            "content": "Suggest two mini-projects to practice Python and APIs.",
        },
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "project_suggestions",
            "strict": True,
            "schema": project_schema,
        }
    },
)

json.loads(json_schema_response.output_text)


{'projects': [{'title': 'Weather Dashboard',
   'difficulty': 'beginner',
   'goal': 'Create a simple web application that fetches and displays weather information from an external API based on user input.'},
  {'title': 'Recipe Finder',
   'difficulty': 'intermediate',
   'goal': 'Develop a command-line tool that allows users to find recipes based on ingredients by making requests to a recipe API.'}]}

## 4. Recursive Structured Output

A useful advanced case is recursive output, such as a tree of UI components.

In [5]:
class UIType(str, Enum):
    div = "div"
    button = "button"
    header = "header"
    section = "section"
    field = "field"
    form = "form"


class Attribute(BaseModel):
    name: str
    value: str


class UI(BaseModel):
    type: UIType
    label: str
    children: list["UI"]
    attributes: list[Attribute]


UI.model_rebuild()


class UIGenerationResult(BaseModel):
    ui: UI


ui_response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {
            "role": "system",
            "content": "Convert the user request into a small structured component"
            "tree.",
        },
        {
            "role": "user",
            "content": "Create a compact user profile form with a header, two text"
            "fields, and a submit button.",
        },
    ],
    text_format=UIGenerationResult,
)

print(ui_response.output_parsed.model_dump_json(indent=2))


{
  "ui": {
    "type": "form",
    "label": "User Profile Form",
    "children": [
      {
        "type": "header",
        "label": "Profile Header",
        "children": [],
        "attributes": [
          {
            "name": "text",
            "value": "User Profile"
          }
        ]
      },
      {
        "type": "field",
        "label": "First Name Field",
        "children": [],
        "attributes": [
          {
            "name": "type",
            "value": "text"
          },
          {
            "name": "placeholder",
            "value": "Enter First Name"
          }
        ]
      },
      {
        "type": "field",
        "label": "Last Name Field",
        "children": [],
        "attributes": [
          {
            "name": "type",
            "value": "text"
          },
          {
            "name": "placeholder",
            "value": "Enter Last Name"
          }
        ]
      },
      {
        "type": "button",
        "label": "Submit B

## 5. Moderation-Style Classification

Structured outputs are also useful when you want a compact decision object that your application can inspect programmatically.

In [6]:
class Category(str, Enum):
    violence = "violence"
    sexual = "sexual"
    self_harm = "self_harm"


class ContentCompliance(BaseModel):
    is_violating: bool
    category: Category | None
    explanation_if_violating: str | None


compliance_response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {
            "role": "system",
            "content": "Determine whether the user message violates the specified"
            "safety policy and explain briefly.",
        },
        {
            "role": "user",
            "content": "How do I prepare for a job interview?",
        },
    ],
    text_format=ContentCompliance,
)

compliance_response.output_parsed


ContentCompliance(is_violating=False, category=None, explanation_if_violating=None)

## Practical Takeaways

- Use Pydantic parsing when Python ergonomics matter most.
- Use explicit JSON schema when downstream systems require exact JSON.
- Recursive models are possible, but keep the schema small and purposeful.
- Safety still applies: structured output is not a bypass around refusals or policy controls.
- For fuller examples, revisit `02_structured_output_scenarios.py`, `16_recursive_structured_output_scenarios.py`, and `17_moderation_structured_output_scenarios.py`.
